# Cálculo de Scores

In [44]:
import pandas as pd
import numpy as np

In [ ]:
produtos = pd.read_parquet('../dados_sinteticos/produtos.parquet')
interacoes = pd.read_parquet('../dados_sinteticos/itens.parquet')

In [59]:
print(produtos.shape)
produtos.head(3)

(100, 4)


,id_produto,id_marca,id_categoria,oferta
0,P_0,M_2,C_9,5
1,P_1,M_0,C_3,20
2,P_2,M_10,C_9,20


In [60]:
print(interacoes.shape)
interacoes.head(3)

(10000, 3)


,id_produto,id_usuario,deslocamento
0,P_0,U_958,4760.773531
1,P_0,U_981,6289.107176
2,P_0,U_16,2816.495495


A iteração abaixo faz todo o processo uma vez para cada dataframe agrupado por usuário:
* Limita a distância percorrida pelo usuário como 2 vezes seu desvio-padrão acima da média
* Normalizar as distâncias com MaxMin considerando máximo e mínimo do usuário (importante)
* Score da aquisição: cada aquisição vale de 1 a 2
* Soma os scores dos produtos para o usuário
  
O resultado é um dicionário cujas chaves são os usuários e cujos valores são dataframes com uma linha por produto com seu respsctivo score

In [ ]:
# Array com os ids dos produtos
ids_produtos = np.sort(produtos['id_produto'].unique())

# Dicionário vazio que vai guardar, para cada usuário, seu vetor de scores
relacionamentos_usuario = {}

for id_usuario, df_user in interacoes.groupby('id_usuario'):
    df_user = df_user.copy() # Para não modificar o dataframe original

    # A distância máxima é o limiar para outliers; qualquer valor acima será igualado a ele
    # dois desvios-padrão acima da média
    media_dist = df_user['deslocamento'].mean()
    desvio_dist = df_user['deslocamento'].std()
    distancia_maxima = media_dist + 2 * desvio_dist

    # .clip(upper= ) corta os valores acima do teto
    # eles viram exatamente o valor do teto, em vez de serem descartados
    df_user['distancia_tratada'] = df_user['deslocamento'].clip(upper=distancia_maxima)

    # # Normalização MaxMin
    # Os valores relativos são para o usuário da vez
    min_dist = df_user['distancia_tratada'].min()
    max_dist = df_user['distancia_tratada'].max()

    if pd.isna(max_dist) or pd.isna(min_dist):  # Tratamento de nan = 0.0
        df_user['distancia_normalizada'] = 0.0
    elif max_dist == min_dist:                  # Se todas as aquisições do usuário são na mesma distância, todas são 0.0
        df_user['distancia_normalizada'] = 0.0
    else:
        df_user['distancia_normalizada'] = (
            (df_user['distancia_tratada'] - min_dist) / (max_dist - min_dist)
        )
    # # 

    # SCORE: cada aquisição vale 1 mais a distância relativa (normalizada), até o máximo 2
    df_user['score'] = df_user['distancia_normalizada'] + 1

    # Os scores são agrupados por produto:
    # Um produto adquirido 2 vezes na menor distância do usuário tem score 2
    # Um produto adquirito 3 vezes, sendo:
        # 1 na menor distância do usuário = 1
        # 1 na maior distância do usuário = 2
        # 1 na média na distância intermediária = 1.5
        # Score do produto para o usuário = 4.5
    df_user = (
        df_user.groupby('id_produto', as_index=False)['score']
        .sum()
    )

    # # Garantia de que todo usuário terá score para todos os produtos
    df_user = (
        pd.DataFrame({'id_produto': ids_produtos})      # DataFrame com a lista completa de produtos
        .merge(df_user, on='id_produto', how='left')    # join: produtos comprados ganham seu score, os demais ficam com NaN
    )

    df_user['score'] = df_user['score'].fillna(0)       # nan -> 0.0

    df_user = df_user.sort_values('id_produto').reset_index(drop=True)  # reset index + sort: padroniza a ordem dos produtos para todos os usuários
    # #

    # O dataframe é adicionado ao dicionário com a chave correspondente ao usuário
    relacionamentos_usuario[id_usuario] = df_user




In [77]:
# Esta é uma forma estendida da célula abaixo

list_to_df = []

for id_usuario, df_user in relacionamentos_usuario.items():         # Para cada dataframe de usuário
    df_user = df_user.copy()                                            # copia o dataframe por segurança
    df_user = df_user.set_index('id_produto')                           # id_produto se torna o índice do df
    score_produto_usuario = df_user['score']                            # seleciona a coluna com os scores (calculados anteriormente)
    score_produto_usuario = score_produto_usuario.rename(id_usuario)    # troca o nome da coluna "score" para o nome do usuário. É o score do usuário "U_..."

    list_to_df.append(score_produto_usuario)                            # Adiciona a série (coluna) à lista reservada

matriz_produto_usuario_exemplo = pd.concat(list_to_df, axis=1)              # concatena as colunas na lista, formando um dataframe

In [78]:
matriz_produto_usuario = pd.concat(
    [
        df_user.set_index('id_produto')['score'].rename(id_usuario)
        for id_usuario, df_user in relacionamentos_usuario.items()
    ],
    axis=1
).sort_index()

In [ ]:
# Agora que produto não precisa mais ser referência, ele pode voltar a ser uma coluna
matriz_produto_usuario = matriz_produto_usuario.reset_index()

In [84]:
matriz_produto_usuario

,id_produto,U_0,U_1,U_10,U_100,U_101,U_102,U_103,U_104,U_105,...,U_99,U_990,U_991,U_992,U_993,U_994,U_995,U_997,U_998,U_999
0,P_0,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
1,P_1,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
2,P_10,0.0,0.00000,0.0,0.0,0.0,0.0,6.331518,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
3,P_11,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
4,P_12,0.0,2.42357,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,P_95,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,1.994154,0.0,...,0.000000,0.0,0.0,0.0,0.0,3.245922,0.0,0.000000,0.0,0.0
96,P_96,0.0,0.00000,0.0,1.0,0.0,0.0,0.000000,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
97,P_97,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,2.385233,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,1.556020,0.0,0.0
98,P_98,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,2.762232,0.0,0.0,0.0,0.0,0.000000,0.0,7.914848,0.0,0.0


In [85]:
matriz_produto_usuario.to_csv('../dados_sinteticos/dados_tratatos/matriz_produto_usuario.csv', index=False)

A matriz_produto_usuario é a matriz de interações do sistema de recomendação: linhas são produtos, colunas são usuários, e cada célula é o score daquele usuário para aquele produto. É ela que alimenta o KNN.